# Manual traces of compiled circuits

Compare the manual Craft forward pass with the assembled Tracr model for
the twelve programs and the focused A/B examples. This notebook imports
`raspformer` directly and can run independently of `programs.ipynb`.

In [1]:
import os
from pathlib import Path

start = Path.cwd().resolve()
PROJECT_ROOT = next(
    root
    for root in (start, *start.parents)
    if (root / "raspformer").is_dir() and (root / "notebooks").is_dir()
)
os.chdir(PROJECT_ROOT)

import numpy as np
import pandas as pd
from IPython.display import display

from raspformer.compiler import CompilerConfig, run_trace_study
from raspformer.programs import build_programs

## Configure and run

Embeddings, output scores, and decoded outputs must agree within tolerance.
Intermediate residual differences are retained as diagnostics, matching the
original study; a warning is not full residual equivalence.

In [2]:
config = CompilerConfig(
    vocab=tuple(range(-2, 6)),
    max_seq_len=6,
    causal=False,
    bos="BOS",
    pad="PAD",
    mlp_exactness=100,
    rel_tol=1e-4,
    abs_tol=1e-4,
)
samples = [
    [1, 2, 1],
    [-2, 0, 3, 1],
    [5],
    [0, 0, 0, 0],
    [5, 4, 3, 2, 1, 0],
]

OUTPUT_ROOT = PROJECT_ROOT / "results" / "compile"
FOCUS_PROGRAM = "B_histogram_then_repeated_class"
FOCUS_SAMPLE = 0
PROGRAMS = build_programs(config.max_seq_len, include_examples=True)

In [3]:
result = run_trace_study(PROGRAMS, config, samples, OUTPUT_ROOT)
evidence = result["evidence"]
display(pd.DataFrame(evidence["programs"]))
print(f"Saved traces: {result['run_dir']}")

P01_absolute: 5 output checks; 0 residual warnings


P02_index_parity: 5 output checks; 0 residual warnings


P03_increment_by_index: 5 output checks; 0 residual warnings


P04_first_element: 5 output checks; 0 residual warnings


P05_histogram: 5 output checks; 0 residual warnings
P06_count_greater_than: 5 output checks; 0 residual warnings


...

P11_check_palindrome: 5 output checks; 0 residual warnings


P12_sorting: 5 output checks; 0 residual warnings
A_prev_token_class: 5 output checks; 0 residual warnings


B_histogram_then_repeated_class: 5 output checks; 0 residual warnings


,program,inputs_checked,max_abs_score_error,residual_max_abs_error,residual_mismatches,status
0,P01_absolute,5,0.000000e+00,0.000000e+00,0,PASS
1,P02_index_parity,5,0.000000e+00,0.000000e+00,0,PASS
2,P03_increment_by_index,5,0.000000e+00,0.000000e+00,0,PASS
3,P04_first_element,5,2.102257e-08,2.102257e-08,0,PASS
4,P05_histogram,5,1.907349e-06,1.907349e-06,0,PASS
5,P06_count_greater_than,5,1.907349e-06,1.907349e-06,0,PASS
6,P07_sum_with_next,5,1.668511e-08,1.907349e-06,0,PASS
7,P08_pairwise_sum,5,1.668511e-08,1.907349e-06,0,PASS
8,P09_check_increasing,5,0.000000e+00,1.907349e-06,0,PASS
9,P10_rotate_left,5,1.668511e-08,1.907349e-06,0,PASS


Saved traces: /Users/raghul/RaspFormer/results/compile/20261006T062048_c56abbc7


## Inspect one trace

Active Craft blocks use the assembled model's attention scaling and causal
setting. The JSON artifact retains full residual arrays and output scores.
Change the program or sample above to inspect another circuit.

In [4]:
examples = [row for row in evidence["traces"] if row["program"] == FOCUS_PROGRAM]
example = examples[FOCUS_SAMPLE]
print(f"Input: {example['tokens']}")
print(f"RASP: {example['rasp_output']}; decoded model: {example['model_decoded']}")
display(
    pd.DataFrame(
        [
            {
                "block": row["module"],
                "residual_shape": np.shape(row["residual"]),
                "output_scores": row["output_scores"],
            }
            for row in example["block_trace"]
        ]
    )
)
print(f"Maximum absolute output-score error: {example['max_abs_score_error']:.3g}")

Input: [1, 2, 1]
RASP: [1, 0, 1]; decoded model: [1, 0, 1]


,block,residual_shape,output_scores
0,transformer/layer_0/attn,"(4, 27)","[[0.0, 0.0], [0.0, 0.0], [0.0, 0.0], [0.0, 0.0]]"
1,transformer/layer_0/mlp,"(4, 27)","[[0.0, 0.0], [0.0, 0.0], [0.0, 0.0], [0.0, 0.0]]"
2,transformer/layer_1/mlp,"(4, 27)","[[0.0, 0.0], [0.0, 1.0], [1.0, 0.0], [0.0, 1.0]]"


Maximum absolute output-score error: 0


## Inspect residual diagnostics

A passing output check does not guarantee every unused residual lane agrees.
Resolve recorded discrepancies before claiming complete circuit equivalence.

In [5]:
warnings = pd.DataFrame(evidence["residual_warnings"])
if warnings.empty:
    print("No residual discrepancies at the configured tolerance.")
else:
    display(warnings)
print(
    f"{len(evidence['traces'])} strict output checks; {len(warnings)} residual diagnostics."
)

No residual discrepancies at the configured tolerance.
70 strict output checks; 0 residual diagnostics.
